# 영상 프레임으로 특정 박스 분류하기

영상에서 프레임을 추출한 후 다음 두 클래스로 나누어 학습합니다.

- `target`: 제거할 박스 한 종류
- `normal`: 통과시킬 나머지 박스 세 종류

실습 순서: **영상 프레임 추출 → 두 폴더로 분류 → 모델 학습 → 예측**

> ROI는 나중에 사용할 선택 기능입니다. 기본 실습에서는 전체 프레임을 사용합니다.

In [1]:
# 필요한 라이브러리
# 설치가 필요하면 다음 줄의 주석을 풀고 한 번만 실행하세요.
%pip install opencv-python tensorflow
# from pathlib import Path
# import shutil
# import cv2
# # 선택 실습(ROI 미리보기)에서 사용: 
# import matplotlib.pyplot as plt
# import tensorflow as tf
# from tensorflow import keras

# print('TensorFlow:', tf.__version__)

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 14.6 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.2/572.2 MB 15.1 MB/s  0:00:35m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 15.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 15.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 12.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 14.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 13.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [tensorflow]2 [tensorflow]on]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart

In [17]:
from pathlib import Path

# 1. 경로와 실습값 설정
PROJECT_DIR = Path(r'D:\code\my_ready\my_FA\faio_opc_lec')
VIDEO_PATH = Path(r'D:\code\my_ready\my_FA\data_set\dataset.mp4')
BOX_DIR = PROJECT_DIR / 'box_dataset'
RAW_DIR = BOX_DIR / 'raw'        # 영상에서 추출한 원본 프레임
DATA_DIR = BOX_DIR / 'labeled'   # 사람이 분류한 학습 데이터

# 선택 실습(ROI): 나중에 사용할 때 아래 좌표의 주석을 푸세요.
ROI_X, ROI_Y = 1000, 360
ROI_W, ROI_H = 380, 440
SAMPLE_SECONDS = 0.1  # 0.5초마다 한 장 추출

assert VIDEO_PATH.exists(), f'영상을 찾을 수 없습니다: {VIDEO_PATH}'
for class_name in ['normal', 'target']:
    (DATA_DIR / class_name).mkdir(parents=True, exist_ok=True)

print('영상:', VIDEO_PATH)
print('학습 폴더:', DATA_DIR)

AssertionError: 영상을 찾을 수 없습니다: D:\code\my_ready\my_FA\data_set\dataset.mp4

In [11]:
# 선택 실습: ROI 영역 확인(현재는 전부 주석 처리)
cap = cv2.VideoCapture(str(VIDEO_PATH))
ok, frame = cap.read()
cap.release()
preview = frame.copy()
cv2.rectangle(preview, (ROI_X, ROI_Y), (ROI_X + ROI_W, ROI_Y + ROI_H), (0, 255, 0), 3)
plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis('off');

print('기본 실습에서는 ROI 없이 전체 프레임을 사용합니다.')

NameError: name 'cv2' is not defined

In [12]:
# 3. 영상에서 프레임 추출
# box_dataset/raw만 새로 만듭니다. 사람이 분류한 labeled 폴더는 삭제하지 않습니다.
shutil.rmtree(RAW_DIR, ignore_errors=True)
RAW_DIR.mkdir(parents=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), '영상을 열 수 없습니다.'
fps = cap.get(cv2.CAP_PROP_FPS)
interval = max(1, round(fps * SAMPLE_SECONDS))
frame_no = saved = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break
    if frame_no % interval == 0:
        # image = frame  # 기본 실습: 전체 프레임 사용
        # 선택 실습(ROI): 아래 줄을 사용할 때 위 줄은 주석 처리하세요.
        image = frame[ROI_Y:ROI_Y + ROI_H, ROI_X:ROI_X + ROI_W]
        cv2.imwrite(str(RAW_DIR / f'frame_{frame_no:06d}.jpg'), image)
        saved += 1
    frame_no += 1

cap.release()
print(f'{saved:,}장 추출 완료 → {RAW_DIR}')

NameError: name 'shutil' is not defined

## 4. 학생 실습: 추출한 프레임에 정답 붙이기

`box_dataset/raw`의 이미지를 확인하여 아래 폴더로 **복사**합니다.

```text
box_dataset/
├── raw/                # 영상에서 추출한 프레임
└── labeled/
    ├── normal/         # 통과시킬 나머지 박스 3종
    └── target/         # 제거할 박스 1종
```

박스가 잘 보이는 사진만 사용하고 빈 화면은 제외하세요. `normal`에는 나머지 세 종류를 골고루 넣으세요.

In [13]:
# 5. 학습 데이터 불러오기: 80% 학습, 20% 검증
IMG_SIZE = (160, 160)
common = dict(directory=DATA_DIR, validation_split=0.2, seed=42, image_size=IMG_SIZE, batch_size=32)
train_ds = keras.utils.image_dataset_from_directory(subset='training', **common)
val_ds = keras.utils.image_dataset_from_directory(subset='validation', **common)
class_names = train_ds.class_names

assert class_names == ['normal', 'target'], f'폴더 이름을 확인하세요: {class_names}'
print('클래스 번호:', dict(enumerate(class_names)))

NameError: name 'keras' is not defined

In [14]:
# 6. MobileNetV2 이진 분류 모델 학습
base = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

model = keras.Sequential([
    keras.layers.Rescaling(1./127.5, offset=-1),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')  # target일 확률
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
stop = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[stop])

loss, acc = model.evaluate(val_ds, verbose=0)
print(f'검증 정확도: {acc:.2%}')
model.save(PROJECT_DIR / 'target_box_model.keras')

NameError: name 'keras' is not defined

In [15]:
# 정답이 target인 이미지로 테스트
TEST_IMAGE = next(
    (DATA_DIR / 'target').glob('*.jpg')
)

img = keras.utils.load_img(
    TEST_IMAGE,
    target_size=IMG_SIZE
)
x = keras.utils.img_to_array(img)[None, ...]

target_probability = float(
    model.predict(x, verbose=0)[0][0]
)

print('테스트 이미지:', TEST_IMAGE.name)
print(f'target 확률: {target_probability:.2%}')
print(
    '판정:',
    '제거' if target_probability >= 0.8 else '통과'
)

StopIteration: 